In [1]:
import spacy
import spacy_experimental

/Users/ameliemajor/Desktop/NLPGenderBias/.venv_coref/lib/python3.10/site-packages/spacy/cli/info.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:

!pip show spacy

Name: spacy
Version: 3.4.4
Summary: Industrial-strength Natural Language Processing (NLP) in Python
Home-page: https://spacy.io
Author: Explosion
Author-email: contact@explosion.ai
License: MIT
Location: /Users/ameliemajor/Desktop/NLPGenderBias/.venv_coref/lib/python3.10/site-packages
Requires: catalogue, cymem, jinja2, langcodes, murmurhash, numpy, packaging, pathy, preshed, pydantic, requests, setuptools, smart-open, spacy-legacy, spacy-loggers, srsly, thinc, tqdm, typer, wasabi
Required-by: en-coreference-web-trf, spacy-experimental, spacy-transformers


In [3]:

nlp = spacy.load("en_coreference_web_trf")

/Users/ameliemajor/Desktop/NLPGenderBias/.venv_coref/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
doc = nlp("The cats were startled by the dog as it growled at them.")

/Users/ameliemajor/Desktop/NLPGenderBias/.venv_coref/lib/python3.10/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


In [5]:

print(doc.spans)

{'coref_clusters_1': [the dog, it], 'coref_clusters_2': [The cats, them]}


In [1]:
import pandas as pd

df = pd.read_csv("data/political_guardian_articles.csv")

brexit_mask = (
    (df["webTitle"].str.contains("brexit", case=False, na=False))
)



brexit_df = df.loc[brexit_mask].copy().dropna(subset=["bodyContent"])
print(f"Articles found: {len(brexit_df)}")
brexit_df["word_count"] = brexit_df["bodyContent"].str.split().str.len()


brexit_articles = brexit_df["bodyContent"]
brexit_articles = brexit_articles.fillna("").astype(str).head(100)

Articles found: 2392


In [7]:
import re
import pandas as pd
import spacy
from collections import defaultdict
from pathlib import Path

names = pd.read_csv("data/mps_frequency_cropped_1990_2024.csv")["name"].tolist()
# ----------------------------
# 1) Load your pipeline
# ----------------------------
# If you truly have this installed, keep it.
# Otherwise, replace with e.g. spacy.load("en_core_web_sm") and note:
#   - you will NOT have coref clusters (doc.spans["coref_clusters_*"])
nlp = spacy.load("en_coreference_web_trf")

# ----------------------------
# 2) Target/alias helpers
# ----------------------------
def _name_to_pattern(name: str) -> re.Pattern:
    """
    Turn 'Theresa May' into a forgiving regex:
      - case-insensitive
      - flexible whitespace
      - word-bounded
    """
    parts = [re.escape(p) for p in name.strip().split()]
    return re.compile(r"\b" + r"\s+".join(parts) + r"\b", re.I)

def build_targets(
    mp_names: list[str],
    aliases: dict[str, list[str]] | None = None
) -> dict[str, re.Pattern]:
    """
    mp_names: ["Theresa May", "Boris Johnson", ...]
    aliases:  {"Boris Johnson": ["Alexander Johnson", "BoJo"], ...}
      - Each canonical MP gets one compiled pattern matching:
        canonical name OR any alias.
    """
    aliases = aliases or {}
    targets = {}
    for canon in mp_names:
        variants = [canon] + aliases.get(canon, [])
        pats = [_name_to_pattern(v).pattern for v in variants]  # take raw pattern strings
        # Combine with alternation, keep flags via re.I on compile
        combo = r"(?:" + r"|".join(pats) + r")"
        targets[canon] = re.compile(combo, re.I)
    return targets

# ----------------------------
# 3) Coref-aware snippet extraction
# ----------------------------
def entity_snippets(text: str, sent_window: int = 0, targets: dict[str, re.Pattern] | None = None):
    """
    Returns dict: {canon_name: [snippet, ...], ...}
    - Uses coref clusters from en_coreference_web_trf (doc.spans["coref_clusters_*"])
    - Keeps only clusters that contain an explicit name/alias match.
    - Adds snippets around each cluster mention, deduped by sentence window.
    """
    if targets is None:
        raise ValueError("targets must be provided (use build_targets).")

    doc = nlp(text)
    sents = list(doc.sents)

    # token index -> sentence index
    tok2sent = {}
    for si, s in enumerate(sents):
        for t in range(s.start, s.end):
            tok2sent[t] = si

    # Collect clusters
    clusters = [
        spangroup for key, spangroup in doc.spans.items()
        if key.startswith("coref_clusters_")
    ]

    out = {canon: [] for canon in targets}
    seen = {canon: set() for canon in targets}  # dedupe by (lo, hi)

    for spangroup in clusters:
        if not spangroup:
            continue

        # Which target(s) does this cluster belong to (by explicit name/alias mention)?
        hit_targets = [
            canon for canon, pat in targets.items()
            if any(pat.search(m.text) for m in spangroup)
        ]
        if not hit_targets:
            continue

        # Add snippets for each mention in the cluster (incl pronouns)
        for mention in spangroup:
            si = tok2sent.get(mention.start)
            if si is None:
                continue

            lo = max(0, si - sent_window)
            hi = min(len(sents), si + sent_window + 1)
            snippet = " ".join(s.text.strip() for s in sents[lo:hi]).strip()

            span_key = (lo, hi)
            for canon in hit_targets:
                if span_key in seen[canon]:
                    continue
                seen[canon].add(span_key)
                out[canon].append(snippet)

    return out

# ----------------------------
# 4) Batch processing utilities
# ----------------------------
def snippets_to_rows(articles, targets: dict[str, re.Pattern], sent_window: int = 1):
    """
    articles: iterable of article texts (Series/list)
    Returns list[dict] rows for a tidy dataframe.
    """
    rows = []
    for article_idx, body_text in enumerate(articles):
        print(f"Processing article {article_idx}...")
        by_entity = entity_snippets(body_text, sent_window=sent_window, targets=targets)
        for person, snippets in by_entity.items():
            for snip_idx, snippet in enumerate(snippets, 1):
                rows.append({
                    "person": person,
                    "article_index": article_idx,
                    "snippet_index": snip_idx,
                    "snippet": snippet,
                })
    return rows

def print_top_snippets(df: pd.DataFrame, max_per_person: int = 3):
    """
    Pretty print up to N snippets per person from the tidy dataframe.
    """
    for person, grp in df.groupby("person", sort=True):
        grp = grp.sort_values(["article_index", "snippet_index"])
        top = grp.head(max_per_person)
        if top.empty:
            continue
        print(f"\nEntity: {person}")
        for i, snip in enumerate(top["snippet"].tolist(), 1):
            print(f"  {i}. {snip}")

def save_per_person_csv(df: pd.DataFrame, out_dir: str | Path):
    """
    Write one CSV per person, automatically.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for person, grp in df.groupby("person", sort=True):
        safe = re.sub(r"[^a-z0-9]+", "_", person.lower()).strip("_")
        grp.to_csv(out_dir / f"{safe}_snippets.csv", index=False)

# ----------------------------
# 5) Example usage
# ----------------------------
MP_NAMES = names

ALIASES = {
    # optional: only if you need them
    "Boris Johnson": ["BoJo", "Alexander Johnson"],
}

TARGETS = build_targets(MP_NAMES, aliases=ALIASES)
print("Compiled target patterns:")
# brexit_articles can be a pandas Series, list of strings, etc.
# If it's a dataframe column, pass brexit_articles["body"] (for example).

rows = snippets_to_rows(brexit_articles, targets=TARGETS, sent_window=1)
df_out = pd.DataFrame(rows)

print_top_snippets(df_out, max_per_person=3)

# One combined CSV (all MPs)
df_out.to_csv("all_mp_snippets.csv", index=False)

# Or per-person CSVs
save_per_person_csv(df_out, out_dir="mp_snippets")


Compiled target patterns:
Processing article 0...


/Users/ameliemajor/Desktop/NLPGenderBias/.venv_coref/lib/python3.10/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


Processing article 1...
Processing article 2...
Processing article 3...
Processing article 4...
Processing article 5...
Processing article 6...
Processing article 7...
Processing article 8...
Processing article 9...
Processing article 10...
Processing article 11...
Processing article 12...
Processing article 13...
Processing article 14...
Processing article 15...
Processing article 16...
Processing article 17...
Processing article 18...
Processing article 19...
Processing article 20...
Processing article 21...
Processing article 22...
Processing article 23...
Processing article 24...
Processing article 25...
Processing article 26...
Processing article 27...
Processing article 28...
Processing article 29...
Processing article 30...
Processing article 31...
Processing article 32...
Processing article 33...
Processing article 34...
Processing article 35...
Processing article 36...
Processing article 37...
Processing article 38...
Processing article 39...
Processing article 40...
Processin

In [2]:
df = pd.read_csv("all_mp_snippets.csv", index_col=False)
print(len(df['person'].unique()))

22
